In [1]:
import mlflow
# Step 1: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://184.72.71.39:5000/")

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='s3://comment-analysis-bucket-994/6', creation_time=1788963985769, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1788963985769, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [4]:
df = pd.read_csv('../data/processed/processed_comments.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Remove missing values
df = df.dropna(subset=['category', 'clean_comment'])

ngram_range = (1, 3)
max_features = 1000

# Final train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Validation split for Optuna
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# TF-IDF fitted only on training data
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_inner_vec = vectorizer.fit_transform(X_train_inner)
X_val_vec = vectorizer.transform(X_val)

# SMOTE only on inner training data
smote = SMOTE(random_state=42)

X_train_inner_vec, y_train_inner = smote.fit_resample(
    X_train_inner_vec,
    y_train_inner
)


# Optuna objective for Random Forest
def objective_rf(trial):

    n_estimators = trial.suggest_int(
        'n_estimators',
        50,
        300
    )

    max_depth = trial.suggest_int(
        'max_depth',
        3,
        20
    )

    min_samples_split = trial.suggest_int(
        'min_samples_split',
        2,
        20
    )

    min_samples_leaf = trial.suggest_int(
        'min_samples_leaf',
        1,
        20
    )

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_inner_vec, y_train_inner)

    y_pred = model.predict(X_val_vec)

    return accuracy_score(y_val, y_pred)


# Run Optuna
study = optuna.create_study(direction="maximize")

study.optimize(
    objective_rf,
    n_trials=30
)

best_params = study.best_params

print("Best parameters:", best_params)
print("Best validation accuracy:", study.best_value)


# Train final model using all training data
final_vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)

smote = SMOTE(random_state=42)

X_train_vec, y_train = smote.fit_resample(
    X_train_vec,
    y_train
)

best_model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train_vec, y_train)

y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", accuracy)


# Log results in MLflow
with mlflow.start_run():

    mlflow.set_tag(
        "mlflow.runName",
        "RandomForest_SMOTE_TFIDF_Trigrams"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    mlflow.log_param("algo_name", "RandomForest")
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("n_trials", 30)

    mlflow.log_params(best_params)

    mlflow.log_metric("accuracy", accuracy)

    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    mlflow.sklearn.log_model(
        best_model,
        name="RandomForest_model"
    )

[I 2026-09-09 18:39:31,020] A new study created in memory with name: no-name-77349ba9-d017-4696-84cb-64a344d88df0
[I 2026-09-09 18:39:31,786] Trial 0 finished with value: 0.6742243436754176 and parameters: {'n_estimators': 126, 'max_depth': 13, 'min_samples_split': 2, 'min_samples_leaf': 19}. Best is trial 0 with value: 0.6742243436754176.
[I 2026-09-09 18:39:33,241] Trial 1 finished with value: 0.643709512444596 and parameters: {'n_estimators': 263, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 19}. Best is trial 0 with value: 0.6742243436754176.
[I 2026-09-09 18:39:34,294] Trial 2 finished with value: 0.6126832594613024 and parameters: {'n_estimators': 139, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 15}. Best is trial 0 with value: 0.6742243436754176.
[I 2026-09-09 18:39:34,909] Trial 3 finished with value: 0.6392771905898398 and parameters: {'n_estimators': 56, 'max_depth': 6, 'min_samples_split': 20, 'min_samples_leaf': 18}. Best is trial 0 with val

Best parameters: {'n_estimators': 160, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 1}
Best validation accuracy: 0.6968973747016707
Final accuracy: 0.6852584208373108
🏃 View run RandomForest_SMOTE_TFIDF_Trigrams at: http://184.72.71.39:5000/#/experiments/6/runs/92f531ae50c0476798be2ff7ba6f6345
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/6
